In [2]:
import arcpy
from arcpy import env
import os
import numpy as np
from arcgis import GIS
from arcgis.features import GeoAccessor
from arcgis.features import GeoSeriesAccessor
import pandas as pd

arcpy.env.overwriteOutput = True
arcpy.env.parallelProcessingFactor = "90%"

# show all columns
pd.options.display.max_columns = None

# pd.pivot_table(df, values='a', index='b', columns='c', aggfunc='sum', fill_value=0)
# pd.DataFrame.spatial.from_featureclass(???)  
# df.spatial.to_featureclass(location=???,sanitize_columns=False)  

# gsa = arcgis.features.GeoSeriesAccessor(df['SHAPE'])  
# df['AREA'] = gsa.area  # KNOW YOUR UNITS

In [ ]:
## spatial join
# target_features = ?
# join_features = ?
# output_features = os.path.join(gdb, ?)

# fieldmappings = arcpy.FieldMappings()
# fieldmappings.addTable(target_features)
# fieldmappings.addTable(join_features)

# # variable
# fieldindex = fieldmappings.findFieldMapIndex(?)
# fieldmap = fieldmappings.getFieldMap(fieldindex)
# fieldmap.mergeRule = 'Sum'
# fieldmappings.replaceFieldMap(fieldindex, fieldmap)

# sj = arcpy.SpatialJoin_analysis(target_features, join_features, output_features,'JOIN_ONE_TO_ONE', "KEEP_ALL", fieldmappings, match_option="INTERSECT")
# sj_df = pd.DataFrame.spatial.from_featureclass(sj[0]).copy()

In [ ]:
# fill NA values in Spatially enabled dataframes (ignores SHAPE column)
def fill_na_sedf(df_with_shape_column, fill_value=0):
    if 'SHAPE' in list(df_with_shape_column.columns):
        cols_to_fill = df_with_shape_column.columns.difference(['SHAPE'])
        df_with_shape_column[cols_to_fill] = df_with_shape_column[cols_to_fill].fillna(fill_value)
        return df_with_shape_column
    else:
        raise Exception("Dataframe does not include 'SHAPE' column")

In [ ]:
outputs = ['.\\Outputs', "scratch.gdb", 'results.gdb']

if not os.path.exists(outputs[0]):
    os.makedirs(outputs[0])

gdb = os.path.join(outputs[0], outputs[1])
gdb2 = os.path.join(outputs[0], outputs[2])

if not arcpy.Exists(gdb):
    arcpy.CreateFileGDB_management(outputs[0], outputs[1])

if not arcpy.Exists(gdb2):
    arcpy.CreateFileGDB_management(outputs[0], outputs[2])

In [4]:
# read in 2050 forecast 
se2050 = pd.read_csv(r"E:\Projects\REMM-v3.0\TDM\_TDMv9.0.0_REMM\1_Inputs\2_SEData\REMM\SE_2050.csv").rename({';TAZID':'TAZID'},axis=1)
se2050.columns

Index(['TAZID', 'CO_TAZID', 'TOTHH', 'HHPOP', 'HHSIZE', 'TOTEMP', 'RETEMP',
       'INDEMP', 'OTHEMP', 'ALLEMP', 'RETL', 'FOOD', 'MANU', 'WSLE', 'OFFI',
       'GVED', 'HLTH', 'OTHR', 'FM_AGRI', 'FM_MING', 'FM_CONS', 'HBJ',
       'AVGINCOME', 'Enrol_Elem', 'Enrol_Midl', 'Enrol_High', 'CO_FIPS',
       'CO_NAME'],
      dtype='object')

In [5]:
hafb_taz = ['hafb', [629,634, 647,630,631,626,628,623,527,540,541]]
daybreak_taz = ['daybreak', [1965,1966,1967,1974,1975,1994,1995,1996,2009]]
the_point_taz = ['the_point', [2138,2140,2141,2149]]
holiday_hills_taz = ['holiday_hills', [1690]]
olympia_hills_taz = ['olympia_hills', [932,933]]

areas = [hafb_taz,daybreak_taz,the_point_taz,holiday_hills_taz,olympia_hills_taz]

In [6]:
for area in areas:
    se2050_area = se2050[se2050['TAZID'].isin(area[1])].copy()
    se2050_area_summary = se2050_area.groupby('TAZID', as_index=False)[['TOTHH', 'INDEMP','OTHEMP', 'RETEMP','TOTEMP']].sum()
    se2050_area_summary.columns = ['TAZID','HH', 'IND','OFF', 'RET','TOTAL_JOBS']
    print(f"\n{area[0].upper()} SUMMARY")
    display(se2050_area_summary)


HAFB SUMMARY


,TAZID,HH,IND,OFF,RET,TOTAL_JOBS
0,527,0.0,246.0,206.0,0.0,452.0
1,540,0.0,2.0,87.0,0.0,89.0
2,541,0.0,1577.0,2234.0,0.0,3811.0
3,623,0.0,976.0,3284.0,3.0,4263.0
4,626,0.0,0.0,801.0,3.0,804.0
5,628,0.0,13.0,4077.0,10.0,4100.0
6,629,0.0,429.0,1482.0,0.0,1911.0
7,630,0.0,308.0,5240.0,0.0,5548.0
8,631,0.0,252.0,8342.0,0.0,8594.0
9,634,46.0,21.0,13656.0,283.0,13960.0



DAYBREAK SUMMARY


,TAZID,HH,IND,OFF,RET,TOTAL_JOBS
0,1965,3.0,33.0,287.0,19.0,339.0
1,1966,49692.0,321.0,4460.0,551.0,5332.0
2,1967,2778.0,47.0,1450.0,231.0,1728.0
3,1974,2267.0,172.0,1991.0,136.0,2299.0
4,1975,3732.0,2.0,152.0,13.0,167.0
5,1994,1067.0,244.0,2131.0,148.0,2523.0
6,1995,1266.0,667.0,7246.0,809.0,8722.0
7,1996,1634.0,383.0,3994.0,369.0,4746.0
8,2009,754.0,418.0,4783.0,329.0,5530.0



THE_POINT SUMMARY


,TAZID,HH,IND,OFF,RET,TOTAL_JOBS
0,2138,7770.0,3.0,47.0,23.0,73.0
1,2140,0.0,154.0,6679.0,435.0,7268.0
2,2141,0.0,105.0,5443.0,479.0,6027.0
3,2149,0.0,141.0,7047.0,478.0,7666.0



HOLIDAY_HILLS SUMMARY


,TAZID,HH,IND,OFF,RET,TOTAL_JOBS
0,1690,579.0,185.0,1345.0,264.0,1794.0



OLYMPIA_HILLS SUMMARY


,TAZID,HH,IND,OFF,RET,TOTAL_JOBS
0,932,5903.0,70.0,623.0,578.0,1271.0
1,933,3568.0,146.0,2357.0,268.0,2771.0
